# 동물상 분류 모델 - Colab 실행 노트북

**런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요.

데이터는 구글 드라이브(`MyDrive/faceproj/data`)에 저장되므로 런타임이 끊겨도 남아 있습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/faceproj/data'
RESULTS = '/content/drive/MyDrive/faceproj/results'
REPO_URL = 'https://github.com/jhnxk/face-matching.git'

In [ ]:
!git clone -q $REPO_URL /content/face-matching || (cd /content/face-matching && git pull -q)
%cd /content/face-matching
!pip -q install mediapipe datasets onnx onnxruntime

In [ ]:
!grep -c "mouse" training/common.py    # 0이 나오면 최신 10종 버전이 맞음.

## 1일차 · FairFace 추출 + 얼굴 정렬·크롭 (약 15~30분)

In [ ]:
!python training/prepare_fairface.py --data-dir $DATA --n 1500

In [ ]:
# 크롭 결과 눈으로 확인
import glob, matplotlib.pyplot as plt
from PIL import Image
fs = sorted(glob.glob(f'{DATA}/images/*.jpg'))[:16]
fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for a, f in zip(ax.flat, fs):
    a.imshow(Image.open(f)); a.axis('off')

## 2일차 · 라벨러별 분량 배정
아래 이름을 팀원 이름으로 바꾸세요. 실행 후 드라이브의 `data/labeling/<이름>` 폴더를 각자 내려받아 `labeler/index.html` 로 라벨링합니다.

In [ ]:
!python training/make_assignments.py --data-dir $DATA --labelers 주환 정원 정유 동윤 채하 --pilot 50 --per-image 3

## 2~3일차 · 라벨 합치기
각자 내보낸 `labels_이름.csv` 를 드라이브의 `data/labels_result/` 폴더에 올린 뒤 실행합니다.

먼저 파일럿만 분석해서 기준표를 보완하고, 본 라벨링이 끝나면 전체를 합칩니다.

In [ ]:
!python training/merge_labels.py --data-dir $DATA --labels-glob "$DATA/labels_result/*.csv" --pilot-only

In [ ]:
!python training/merge_labels.py --data-dir $DATA --labels-glob "$DATA/labels_result/*.csv"
!python training/split_dataset.py --data-dir $DATA

## 4일차 · 학습 + 실험 (T4 기준 전체 약 30~60분)
`--exp all` = MLP 1개 + Linear probe 4개(25/50/75/100%) + Fine-tuning 4개

In [ ]:
!python training/train.py --data-dir $DATA --out-dir $RESULTS --exp all

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display
display(pd.read_csv(f'{RESULTS}/summary.csv'))
display(Show(f'{RESULTS}/datasize.png'))
display(Show(f'{RESULTS}/finetune_f100/curves.png'))
display(Show(f'{RESULTS}/finetune_f100/confusion.png'))

## 5일차 · ONNX 변환 → 다운로드
내려받은 `face_type.onnx`, `meta.json` 을 저장소의 `web/model/` 에 넣고 커밋합니다.

In [ ]:
!python training/export_onnx.py --ckpt $RESULTS/finetune_f100/model.pt --out-dir $RESULTS/onnx
from google.colab import files
files.download(f'{RESULTS}/onnx/face_type.onnx'); files.download(f'{RESULTS}/onnx/meta.json')